In [1]:
"""
=============================================================================
  TITAN V2 — PHYSICS-INFORMED SciML TRAINING ENGINE  (NaN-hardened)
  Jet Impingement CFD Surrogate Model

  Architecture : DeepONet-style Branch/Trunk MLP with SwiGLU activations
  Inputs       : 12  (4 Global + 5 Spatial + 2 Skewed + 1 Binary)
  Targets      : 5   (T, P, U, V, W)
  Aux targets  : 2   (Theta_norm, Nu_local)
  Hardware     : GTX 1650 (4 GB VRAM) safe

ROOT CAUSE ANALYSIS — why NaN appeared at epoch 13
────────────────────────────────────────────────────
NaN FIX 1 — AMP SCALER SKIP + STALE GRADIENT ACCUMULATION
  When AMP GradScaler detects inf/nan it skips the optimizer step but
  does NOT zero gradients. Those stale inf/nan values accumulate across
  the next micro-batches → permanent NaN spiral.
  Fix: detect scale reduction after amp_scaler.step(); if skipped,
  immediately zero_grad and reload best checkpoint weights.

NaN FIX 2 — Nu_local OUTLIERS IN AUX TARGET
  Nu_local is zero for ~99% of points and extreme at wall nodes where
  dT→0. After StandardScaler this produces ±50-200 std values in the
  aux tensor. MSE on these → loss spike → exploding gradient.
  Fix: clamp aux targets to ±5 std + use Huber loss (not MSE) for aux.
  Also reduce w_aux 0.20→0.10.

NaN FIX 3 — SWIGLU BRANCH×TRUNK PRODUCT SATURATION
  b * t can produce very large values when both are simultaneously large.
  RMSNorm after the product helps but the product itself already overflows.
  Fix: apply separate RMSNorm to branch and trunk outputs BEFORE product.

NaN FIX 4 — LR TOO HIGH RELATIVE TO WARMUP
  3-epoch warmup over 16M+ rows = full lr=5e-4 reached after millions of
  steps at ~ep 12-13 ReduceLROnPlateau had not yet fired. The model hit
  a sharp valley and exploded.
  Fix: lr 5e-4→2e-4, warmup 3→5 epochs, grad_clip 1.0→0.5.

NaN FIX 5 — NO ROLLBACK ON NaN DETECTION
  When NaN fires the in-memory weights are corrupted. The checkpoint on
  disk is clean but was never reloaded. Final verify used corrupted model.
  Fix: reload best checkpoint immediately on NaN detection. Final verify
  always loads from disk, never from in-memory model.
=============================================================================
"""

import os
import time
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm.auto import tqdm

# ─────────────────────────────────────────────────
#  REPRODUCIBILITY
# ─────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ─────────────────────────────────────────────────
#  HYPERPARAMETERS
# ─────────────────────────────────────────────────
CFG = dict(
    # Paths
    dataset_path    = 'Titan_Engineered_Dataset.pt',
    checkpoint_path = 'Jet_TITAN_V2_Best.pth',
    history_path    = 'training_history.png',
    val_split       = 0.10,

    # Architecture — must match FE pipeline exactly
    n_global        = 4,       # H_D, Re, Heat_Flux, Stanton
    n_spatial       = 8,       # X,Y,Z,R,Dist,Y_norm,RadVel,StagnFlag
    n_targets       = 5,       # T, P, U, V, W
    n_aux           = 2,       # Theta_norm, Nu_local
    width           = 512,

    # Training
    max_epochs      = 200,
    warmup_epochs   = 5,       # NaN FIX 4: 3→5
    micro_batch     = 128,
    accum_steps     = 64,      # effective batch = 8 192
    val_batch       = 1024,
    lr              = 2e-4,    # NaN FIX 4: 5e-4→2e-4
    weight_decay    = 1e-4,
    grad_clip       = 0.5,     # NaN FIX 4: 1.0→0.5

    # Loss weights
    w_temp          = 1.0,
    w_press         = 1.0,
    w_u             = 1.0,
    w_v             = 1.0,
    w_w             = 1.0,
    w_aux           = 0.10,    # NaN FIX 2: 0.20→0.10
    aux_clamp       = 5.0,     # NaN FIX 2: clamp aux targets to ±5 std

    # Early stopping
    patience        = 10,
    lr_factor       = 0.5,
    lr_patience     = 4,
    min_lr          = 1e-6,
)


# ─────────────────────────────────────────────────
#  DATASET
# ─────────────────────────────────────────────────
class JetImpingementDataset(Dataset):
    def __init__(self, X: np.ndarray, T: np.ndarray, A: np.ndarray):
        self.X = torch.from_numpy(X)
        self.T = torch.from_numpy(T)
        self.A = torch.from_numpy(A)
    def __len__(self):           return len(self.T)
    def __getitem__(self, idx):  return self.X[idx], self.T[idx], self.A[idx]
    @property
    def input_dim(self):         return self.X.shape[1]
    @property
    def target_dim(self):        return self.T.shape[1]


# ─────────────────────────────────────────────────
#  ARCHITECTURE
# ─────────────────────────────────────────────────
class SwiGLUBlock(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        hidden    = int(dim * 4 / 3)
        self.norm = nn.RMSNorm(dim)
        self.w1   = nn.Linear(dim, hidden, bias=False)
        self.w2   = nn.Linear(dim, hidden, bias=False)
        self.w3   = nn.Linear(hidden, dim, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.norm(x)
        return x + self.w3(F.silu(self.w1(h)) * self.w2(h))


class TitanSciML(nn.Module):
    """
    DeepONet branch/trunk network.
    x_full[:, 0:4]  -> global   (H_D, Reynolds, Heat_Flux, Stanton_Proxy)
    x_full[:, 4:12] -> spatial  (X, Y, Z, Radius, Dist_Outflow,
                                  Y_norm, Radial_Vel_Mag, Stagnation_Flag)
    output: (B, 7)  -> [:5] main targets, [5:] aux targets
    """
    def __init__(self, n_global: int = 4, n_spatial: int = 8, width: int = 512):
        super().__init__()
        self.n_global  = n_global
        self.n_spatial = n_spatial

        self.branch      = nn.Sequential(nn.Linear(n_global, width),
                                         SwiGLUBlock(width),
                                         SwiGLUBlock(width),
                                         SwiGLUBlock(width))
        # NaN FIX 3: normalise branch before product
        self.branch_norm = nn.RMSNorm(width)

        self.trunk       = nn.Sequential(nn.Linear(n_spatial, width),
                                         SwiGLUBlock(width),
                                         SwiGLUBlock(width),
                                         SwiGLUBlock(width))
        # NaN FIX 3: normalise trunk before product
        self.trunk_norm  = nn.RMSNorm(width)

        self.fusion_norm = nn.RMSNorm(width)

        self.head = nn.Sequential(SwiGLUBlock(width), nn.Linear(width, 7))
        nn.init.uniform_(self.head[-1].weight, -0.01, 0.01)
        nn.init.zeros_(self.head[-1].bias)

    def forward(self, x_full: torch.Tensor) -> torch.Tensor:
        g = x_full[:, :self.n_global]
        s = x_full[:, self.n_global: self.n_global + self.n_spatial]
        # NaN FIX 3: normalise both sides before product
        b     = self.branch_norm(self.branch(g))
        t     = self.trunk_norm(self.trunk(s))
        fused = self.fusion_norm(b * t)
        return self.head(fused)


# ─────────────────────────────────────────────────
#  LOSS
# ─────────────────────────────────────────────────
class PhysicsInformedLoss(nn.Module):
    """
    Weighted MSE for main targets.
    Huber loss for aux targets (robust to Nu_local outliers).
    NaN FIX 2: aux targets clamped + Huber instead of MSE.
    """
    def __init__(self, cfg: dict):
        super().__init__()
        self.register_buffer('target_weights', torch.tensor(
            [cfg['w_temp'], cfg['w_press'], cfg['w_u'], cfg['w_v'], cfg['w_w']],
            dtype=torch.float32))
        self.w_aux     = cfg['w_aux']
        self.aux_clamp = cfg['aux_clamp']
        self.n_main    = cfg['n_targets']

    def forward(self, predictions, target_main, target_aux):
        pred_main = predictions[:, :self.n_main]
        pred_aux  = predictions[:, self.n_main:]

        # Weighted MSE — main (all O(1) post-StandardScaler)
        loss_main = ((pred_main - target_main) ** 2 * self.target_weights).mean()

        # NaN FIX 2: clamp extreme aux values, then Huber loss
        ta_clamped = target_aux.clamp(-self.aux_clamp, self.aux_clamp)
        loss_aux   = F.huber_loss(pred_aux, ta_clamped, delta=1.0)

        return loss_main + self.w_aux * loss_aux, loss_main, loss_aux


# ─────────────────────────────────────────────────
#  HELPERS
# ─────────────────────────────────────────────────
def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)
def is_bad(t):       return bool(torch.isnan(t) or torch.isinf(t))

def make_warmup_lambda(n):
    return lambda ep: float(ep + 1) / n if ep < n else 1.0

def load_best(model, path, device):
    if os.path.exists(path):
        model.load_state_dict(torch.load(path, map_location=device, weights_only=True))
        model.train()

def save_plot(history, path):
    n = len(history['train_main'])
    if n == 0: return
    ep = range(1, n + 1)
    fig, ax = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('Titan V2 — Training History', fontsize=14)

    ax[0].plot(ep, history['train_main'], label='Train'); ax[0].plot(ep, history['val_main'], label='Val')
    ax[0].set_title('Primary Loss  T,P,U,V,W'); ax[0].set_yscale('log')
    ax[0].set_xlabel('Epoch'); ax[0].legend(); ax[0].grid(alpha=0.3)

    ax[1].plot(ep, history['train_aux'], label='Train'); ax[1].plot(ep, history['val_aux'], label='Val')
    ax[1].set_title('Aux Physics Loss  (Huber)'); ax[1].set_yscale('log')
    ax[1].set_xlabel('Epoch'); ax[1].legend(); ax[1].grid(alpha=0.3)

    ax[2].plot(ep, history['lr']); ax[2].set_title('LR Schedule')
    ax[2].set_yscale('log'); ax[2].set_xlabel('Epoch'); ax[2].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"[Plot]  Saved → {path}")


# ─────────────────────────────────────────────────
#  MAIN
# ─────────────────────────────────────────────────
def main():
    print("\n" + "="*65)
    print("  TITAN V2  ·  JET IMPINGEMENT SciML  (NaN-hardened)")
    print("="*65)

    device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    use_amp = device.type == 'cuda'
    print(f"[Device]  {device}  |  AMP: {use_amp}")

    # ── Data ─────────────────────────────────────────────────────────
    dataset = torch.load(CFG['dataset_path'], weights_only=False)
    print(f"[Data]  {len(dataset):,} samples  |  in={dataset.input_dim}  tgt={dataset.target_dim}")

    assert dataset.input_dim  == CFG['n_global'] + CFG['n_spatial'], \
        f"Input dim mismatch: {dataset.input_dim} vs {CFG['n_global']+CFG['n_spatial']}"
    assert dataset.target_dim == CFG['n_targets'], \
        f"Target dim mismatch: {dataset.target_dim} vs {CFG['n_targets']}"

    n_val   = int(CFG['val_split'] * len(dataset))
    n_train = len(dataset) - n_val
    train_ds, val_ds = random_split(dataset, [n_train, n_val],
                                    generator=torch.Generator().manual_seed(SEED))
    print(f"[Data]  Train: {n_train:,}  Val: {n_val:,}")

    train_loader = DataLoader(train_ds, batch_size=CFG['micro_batch'],
                              shuffle=True,  pin_memory=use_amp,
                              num_workers=0, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=CFG['val_batch'],
                              shuffle=False, pin_memory=use_amp, num_workers=0)

    # ── Model ────────────────────────────────────────────────────────
    model = TitanSciML(CFG['n_global'], CFG['n_spatial'], CFG['width']).to(device)
    print(f"[Model]  {count_params(model):,} params  ({count_params(model)/1e6:.2f} M)")

    # ── Loss / Optimiser / Schedulers ────────────────────────────────
    criterion   = PhysicsInformedLoss(CFG).to(device)
    optimizer   = optim.AdamW(model.parameters(),
                               lr=CFG['lr'], weight_decay=CFG['weight_decay'])
    warmup_sch  = optim.lr_scheduler.LambdaLR(
        optimizer, lr_lambda=make_warmup_lambda(CFG['warmup_epochs']))
    plateau_sch = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=CFG['lr_factor'],
        patience=CFG['lr_patience'], min_lr=CFG['min_lr'], verbose=False)
    scaler      = torch.amp.GradScaler('cuda', enabled=use_amp)

    # ── Training state ───────────────────────────────────────────────
    best_val      = float('inf')
    no_improve    = 0
    nan_total     = 0
    history       = dict(train_main=[], train_aux=[], val_main=[], val_aux=[], lr=[])

    print(f"\n[Train]  {CFG['max_epochs']} ep | "
          f"eff-batch {CFG['micro_batch']*CFG['accum_steps']:,} | "
          f"lr={CFG['lr']:.0e} | warmup={CFG['warmup_epochs']} | clip={CFG['grad_clip']}\n")
    t0 = time.time()

    for epoch in range(1, CFG['max_epochs'] + 1):

        # ── TRAIN ────────────────────────────────────────────────────
        model.train()
        main_sum = aux_sum = nan_steps = 0

        pbar = tqdm(enumerate(train_loader), total=len(train_loader),
                    desc=f"Ep {epoch:03d}/{CFG['max_epochs']}", leave=False)
        optimizer.zero_grad()

        for step, (xb, tb, ab) in pbar:
            xb = xb.to(device, non_blocking=True)
            tb = tb.to(device, non_blocking=True)
            ab = ab.to(device, non_blocking=True)

            with torch.amp.autocast('cuda', enabled=use_amp):
                preds              = model(xb)
                loss, l_main, l_aux = criterion(preds, tb, ab)

            # NaN FIX 2: catch NaN loss before backward
            if is_bad(loss):
                nan_steps += 1
                optimizer.zero_grad()
                scaler.update()
                continue

            scaler.scale(loss / CFG['accum_steps']).backward()
            main_sum += l_main.item()
            aux_sum  += l_aux.item()

            is_last  = (step + 1) == len(train_loader)
            is_accum = (step + 1) % CFG['accum_steps'] == 0

            if is_accum or is_last:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip'])

                # NaN FIX 1: detect AMP skip via scale drop
                scale_before = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()

                if scaler.get_scale() < scale_before:
                    # AMP skipped the step — weights are safe but grads are bad
                    nan_steps += 1
                    optimizer.zero_grad()
                    load_best(model, CFG['checkpoint_path'], device)  # NaN FIX 5
                else:
                    optimizer.zero_grad()

            if step % 100 == 0:
                pbar.set_postfix(Main=f'{l_main.item():.4f}',
                                 Aux=f'{l_aux.item():.4f}',
                                 NaN=nan_steps)

        if nan_steps:
            nan_total += nan_steps
            print(f"  ⚠  {nan_steps} NaN steps skipped  (cumulative: {nan_total})")

        valid = max(len(train_loader) - nan_steps, 1)
        avg_tr_main = main_sum / valid
        avg_tr_aux  = aux_sum  / valid

        # ── VALIDATE ─────────────────────────────────────────────────
        model.eval()
        vm_sum = va_sum = bad_batches = 0

        with torch.no_grad():
            for xv, tv, av in val_loader:
                xv = xv.to(device, non_blocking=True)
                tv = tv.to(device, non_blocking=True)
                av = av.to(device, non_blocking=True)
                with torch.amp.autocast('cuda', enabled=use_amp):
                    vp, (_, v_main, v_aux) = model(xv), criterion(model(xv), tv, av)
                if is_bad(v_main):
                    bad_batches += 1
                    continue
                vm_sum += v_main.item()
                va_sum += v_aux.item()

        # NaN FIX 5: full val NaN → roll back, skip epoch
        if bad_batches == len(val_loader):
            print(f"  ✗  Entire val NaN — rolling back to best checkpoint")
            load_best(model, CFG['checkpoint_path'], device)
            continue

        valid_val    = max(len(val_loader) - bad_batches, 1)
        avg_val_main = vm_sum / valid_val
        avg_val_aux  = va_sum / valid_val
        current_lr   = optimizer.param_groups[0]['lr']

        # ── SCHEDULERS ───────────────────────────────────────────────
        if epoch <= CFG['warmup_epochs']:
            warmup_sch.step()
        else:
            plateau_sch.step(avg_val_main)

        # ── HISTORY & LOG ────────────────────────────────────────────
        history['train_main'].append(avg_tr_main)
        history['train_aux' ].append(avg_tr_aux)
        history['val_main'  ].append(avg_val_main)
        history['val_aux'   ].append(avg_val_aux)
        history['lr'        ].append(current_lr)

        print(f"Ep {epoch:03d} | "
              f"Tr {avg_tr_main:.5f}/{avg_tr_aux:.5f} | "
              f"Val {avg_val_main:.5f}/{avg_val_aux:.5f} | "
              f"LR {current_lr:.2e} | "
              f"{(time.time()-t0)/60:.1f} min")

        # ── CHECKPOINT + EARLY STOPPING ──────────────────────────────
        if avg_val_main < best_val:
            best_val   = avg_val_main
            no_improve = 0
            torch.save(model.state_dict(), CFG['checkpoint_path'])
            print(f"  ✦ New best {best_val:.5f} → saved")
        else:
            no_improve += 1
            print(f"  · {no_improve}/{CFG['patience']}")

        if no_improve >= CFG['patience']:
            print(f"\n[Stop]  Early stop at ep {epoch}  best={best_val:.5f}")
            break

    # ── SUMMARY ──────────────────────────────────────────────────────
    save_plot(history, CFG['history_path'])
    print(f"\n{'='*65}")
    print(f"  ✅  DONE  |  Best val main: {best_val:.6f}  |  "
          f"NaN recoveries: {nan_total}  |  "
          f"Time: {(time.time()-t0)/60:.1f} min")
    print(f"{'='*65}\n")

    # ── VERIFY FROM DISK — never from in-memory model (NaN FIX 5) ────
    model_best = TitanSciML(CFG['n_global'], CFG['n_spatial'], CFG['width']).to(device)
    model_best.load_state_dict(
        torch.load(CFG['checkpoint_path'], map_location=device, weights_only=True))
    model_best.eval()
    xv, tv, av = next(iter(val_loader))
    with torch.no_grad():
        out = model_best(xv.to(device))
    assert not torch.isnan(out).any(), "CHECKPOINT HAS NaN — training failed"
    print(f"[Verify]  shape={out.shape}  range=[{out.min():.3f}, {out.max():.3f}]")
    print("[Verify]  ✓  Clean checkpoint ready for inference.")


if __name__ == "__main__":
    main()

c:\Users\choud\anaconda3\envs\Jet\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



  TITAN V2  ·  JET IMPINGEMENT SciML  (NaN-hardened)
[Device]  cuda  |  AMP: True
[Data]  16,600,000 samples  |  in=12  tgt=5
[Data]  Train: 14,940,000  Val: 1,660,000
[Model]  7,348,743 params  (7.35 M)


c:\Users\choud\anaconda3\envs\Jet\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(



[Train]  200 ep | eff-batch 8,192 | lr=2e-04 | warmup=5 | clip=0.5



KeyboardInterrupt: 

In [ ]:
"""
=============================================================================
  TITAN V2 — PHYSICS-INFORMED SciML TRAINING ENGINE  (NaN-hardened)
  Jet Impingement CFD Surrogate Model

  Architecture : DeepONet-style Branch/Trunk MLP with SwiGLU activations
  Inputs       : 12  (4 Global + 5 Spatial + 2 Skewed + 1 Binary)
  Targets      : 5   (T, P, U, V, W)
  Aux targets  : 2   (Theta_norm, Nu_local)
  Hardware     : GTX 1650 (4 GB VRAM) safe

ROOT CAUSE ANALYSIS — why NaN appeared at epoch 13
────────────────────────────────────────────────────
NaN FIX 1 — AMP SCALER SKIP + STALE GRADIENT ACCUMULATION
  When AMP GradScaler detects inf/nan it skips the optimizer step but
  does NOT zero gradients. Those stale inf/nan values accumulate across
  the next micro-batches → permanent NaN spiral.
  Fix: detect scale reduction after amp_scaler.step(); if skipped,
  immediately zero_grad and reload best checkpoint weights.

NaN FIX 2 — Nu_local OUTLIERS IN AUX TARGET
  Nu_local is zero for ~99% of points and extreme at wall nodes where
  dT→0. After StandardScaler this produces ±50-200 std values in the
  aux tensor. MSE on these → loss spike → exploding gradient.
  Fix: clamp aux targets to ±5 std + use Huber loss (not MSE) for aux.
  Also reduce w_aux 0.20→0.10.

NaN FIX 3 — SWIGLU BRANCH×TRUNK PRODUCT SATURATION
  b * t can produce very large values when both are simultaneously large.
  RMSNorm after the product helps but the product itself already overflows.
  Fix: apply separate RMSNorm to branch and trunk outputs BEFORE product.

NaN FIX 4 — LR TOO HIGH RELATIVE TO WARMUP
  3-epoch warmup over 16M+ rows = full lr=5e-4 reached after millions of
  steps at ~ep 12-13 ReduceLROnPlateau had not yet fired. The model hit
  a sharp valley and exploded.
  Fix: lr 5e-4→2e-4, warmup 3→5 epochs, grad_clip 1.0→0.5.

NaN FIX 5 — NO ROLLBACK ON NaN DETECTION
  When NaN fires the in-memory weights are corrupted. The checkpoint on
  disk is clean but was never reloaded. Final verify used corrupted model.
  Fix: reload best checkpoint immediately on NaN detection. Final verify
  always loads from disk, never from in-memory model.
=============================================================================
"""

import os
import time
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm.auto import tqdm

# ─────────────────────────────────────────────────
#  REPRODUCIBILITY
# ─────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ─────────────────────────────────────────────────
#  HYPERPARAMETERS
# ─────────────────────────────────────────────────
CFG = dict(
    # Paths
    dataset_path    = 'Titan_Engineered_Dataset.pt',
    checkpoint_path = 'Jet_TITAN_V2_Best.pth',
    history_path    = 'training_history.png',
    val_split       = 0.10,

    # Architecture — must match FE pipeline exactly
    n_global        = 4,       # H_D, Re, Heat_Flux, Stanton
    n_spatial       = 8,       # X,Y,Z,R,Dist,Y_norm,RadVel,StagnFlag
    n_targets       = 5,       # T, P, U, V, W
    n_aux           = 2,       # Theta_norm, Nu_local
    width           = 512,

    # Training
    max_epochs      = 200,
    warmup_epochs   = 5,       # NaN FIX 4: 3→5
    micro_batch     = 2048,    # SPEED FIX: was 128
    accum_steps     = 4,       # SPEED FIX: was 64 — eff-batch unchanged 2048x4=8192
    val_batch       = 4096,    # SPEED FIX: no grads, push higher
    lr              = 2e-4,    # NaN FIX 4: 5e-4→2e-4
    weight_decay    = 1e-4,
    grad_clip       = 0.5,     # NaN FIX 4: 1.0→0.5

    # Loss weights
    w_temp          = 1.0,
    w_press         = 1.0,
    w_u             = 1.0,
    w_v             = 1.0,
    w_w             = 1.0,
    w_aux           = 0.10,    # NaN FIX 2: 0.20→0.10
    aux_clamp       = 5.0,     # NaN FIX 2: clamp aux targets to ±5 std

    # Early stopping
    patience        = 10,
    lr_factor       = 0.5,
    lr_patience     = 4,
    min_lr          = 1e-6,
)


# ─────────────────────────────────────────────────
#  DATASET
# ─────────────────────────────────────────────────
class JetImpingementDataset(Dataset):
    def __init__(self, X: np.ndarray, T: np.ndarray, A: np.ndarray):
        self.X = torch.from_numpy(X)
        self.T = torch.from_numpy(T)
        self.A = torch.from_numpy(A)
    def __len__(self):           return len(self.T)
    def __getitem__(self, idx):  return self.X[idx], self.T[idx], self.A[idx]
    @property
    def input_dim(self):         return self.X.shape[1]
    @property
    def target_dim(self):        return self.T.shape[1]


# ─────────────────────────────────────────────────
#  ARCHITECTURE
# ─────────────────────────────────────────────────
class SwiGLUBlock(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        hidden    = int(dim * 4 / 3)
        self.norm = nn.RMSNorm(dim)
        self.w1   = nn.Linear(dim, hidden, bias=False)
        self.w2   = nn.Linear(dim, hidden, bias=False)
        self.w3   = nn.Linear(hidden, dim, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.norm(x)
        return x + self.w3(F.silu(self.w1(h)) * self.w2(h))


class TitanSciML(nn.Module):
    """
    DeepONet branch/trunk network.
    x_full[:, 0:4]  -> global   (H_D, Reynolds, Heat_Flux, Stanton_Proxy)
    x_full[:, 4:12] -> spatial  (X, Y, Z, Radius, Dist_Outflow,
                                  Y_norm, Radial_Vel_Mag, Stagnation_Flag)
    output: (B, 7)  -> [:5] main targets, [5:] aux targets
    """
    def __init__(self, n_global: int = 4, n_spatial: int = 8, width: int = 512):
        super().__init__()
        self.n_global  = n_global
        self.n_spatial = n_spatial

        self.branch      = nn.Sequential(nn.Linear(n_global, width),
                                         SwiGLUBlock(width),
                                         SwiGLUBlock(width),
                                         SwiGLUBlock(width))
        # NaN FIX 3: normalise branch before product
        self.branch_norm = nn.RMSNorm(width)

        self.trunk       = nn.Sequential(nn.Linear(n_spatial, width),
                                         SwiGLUBlock(width),
                                         SwiGLUBlock(width),
                                         SwiGLUBlock(width))
        # NaN FIX 3: normalise trunk before product
        self.trunk_norm  = nn.RMSNorm(width)

        self.fusion_norm = nn.RMSNorm(width)

        self.head = nn.Sequential(SwiGLUBlock(width), nn.Linear(width, 7))
        nn.init.uniform_(self.head[-1].weight, -0.01, 0.01)
        nn.init.zeros_(self.head[-1].bias)

    def forward(self, x_full: torch.Tensor) -> torch.Tensor:
        g = x_full[:, :self.n_global]
        s = x_full[:, self.n_global: self.n_global + self.n_spatial]
        # NaN FIX 3: normalise both sides before product
        b     = self.branch_norm(self.branch(g))
        t     = self.trunk_norm(self.trunk(s))
        fused = self.fusion_norm(b * t)
        return self.head(fused)


# ─────────────────────────────────────────────────
#  LOSS
# ─────────────────────────────────────────────────
class PhysicsInformedLoss(nn.Module):
    """
    Weighted MSE for main targets.
    Huber loss for aux targets (robust to Nu_local outliers).
    NaN FIX 2: aux targets clamped + Huber instead of MSE.
    """
    def __init__(self, cfg: dict):
        super().__init__()
        self.register_buffer('target_weights', torch.tensor(
            [cfg['w_temp'], cfg['w_press'], cfg['w_u'], cfg['w_v'], cfg['w_w']],
            dtype=torch.float32))
        self.w_aux     = cfg['w_aux']
        self.aux_clamp = cfg['aux_clamp']
        self.n_main    = cfg['n_targets']

    def forward(self, predictions, target_main, target_aux):
        pred_main = predictions[:, :self.n_main]
        pred_aux  = predictions[:, self.n_main:]

        # Weighted MSE — main (all O(1) post-StandardScaler)
        loss_main = ((pred_main - target_main) ** 2 * self.target_weights).mean()

        # NaN FIX 2: clamp extreme aux values, then Huber loss
        ta_clamped = target_aux.clamp(-self.aux_clamp, self.aux_clamp)
        loss_aux   = F.huber_loss(pred_aux, ta_clamped, delta=1.0)

        return loss_main + self.w_aux * loss_aux, loss_main, loss_aux


# ─────────────────────────────────────────────────
#  HELPERS
# ─────────────────────────────────────────────────
def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)
def is_bad(t):       return bool(torch.isnan(t) or torch.isinf(t))

def make_warmup_lambda(n):
    return lambda ep: float(ep + 1) / n if ep < n else 1.0

def load_best(model, path, device):
    if os.path.exists(path):
        model.load_state_dict(torch.load(path, map_location=device, weights_only=True))
        model.train()

def save_plot(history, path):
    n = len(history['train_main'])
    if n == 0: return
    ep = range(1, n + 1)
    fig, ax = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('Titan V2 — Training History', fontsize=14)

    ax[0].plot(ep, history['train_main'], label='Train'); ax[0].plot(ep, history['val_main'], label='Val')
    ax[0].set_title('Primary Loss  T,P,U,V,W'); ax[0].set_yscale('log')
    ax[0].set_xlabel('Epoch'); ax[0].legend(); ax[0].grid(alpha=0.3)

    ax[1].plot(ep, history['train_aux'], label='Train'); ax[1].plot(ep, history['val_aux'], label='Val')
    ax[1].set_title('Aux Physics Loss  (Huber)'); ax[1].set_yscale('log')
    ax[1].set_xlabel('Epoch'); ax[1].legend(); ax[1].grid(alpha=0.3)

    ax[2].plot(ep, history['lr']); ax[2].set_title('LR Schedule')
    ax[2].set_yscale('log'); ax[2].set_xlabel('Epoch'); ax[2].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"[Plot]  Saved → {path}")


# ─────────────────────────────────────────────────
#  MAIN
# ─────────────────────────────────────────────────
def main():
    print("\n" + "="*65)
    print("  TITAN V2  ·  JET IMPINGEMENT SciML  (NaN-hardened)")
    print("="*65)

    device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    use_amp = device.type == 'cuda'
    print(f"[Device]  {device}  |  AMP: {use_amp}")

    # ── Data ─────────────────────────────────────────────────────────
    dataset = torch.load(CFG['dataset_path'], weights_only=False)
    print(f"[Data]  {len(dataset):,} samples  |  in={dataset.input_dim}  tgt={dataset.target_dim}")

    assert dataset.input_dim  == CFG['n_global'] + CFG['n_spatial'], \
        f"Input dim mismatch: {dataset.input_dim} vs {CFG['n_global']+CFG['n_spatial']}"
    assert dataset.target_dim == CFG['n_targets'], \
        f"Target dim mismatch: {dataset.target_dim} vs {CFG['n_targets']}"

    n_val   = int(CFG['val_split'] * len(dataset))
    n_train = len(dataset) - n_val
    train_ds, val_ds = random_split(dataset, [n_train, n_val],
                                    generator=torch.Generator().manual_seed(SEED))
    print(f"[Data]  Train: {n_train:,}  Val: {n_val:,}")

    train_loader = DataLoader(train_ds, batch_size=CFG['micro_batch'],
                              shuffle=True,  pin_memory=use_amp,
                              num_workers=0, drop_last=True)
                              
    val_loader   = DataLoader(val_ds,   batch_size=CFG['val_batch'],
                              shuffle=False, pin_memory=use_amp, 
                              num_workers=0)

    # ── Model ────────────────────────────────────────────────────────
    model = TitanSciML(CFG['n_global'], CFG['n_spatial'], CFG['width']).to(device)
    print(f"[Model]  {count_params(model):,} params  ({count_params(model)/1e6:.2f} M)")

    # ── Loss / Optimiser / Schedulers ────────────────────────────────
    criterion   = PhysicsInformedLoss(CFG).to(device)
    optimizer   = optim.AdamW(model.parameters(),
                               lr=CFG['lr'], weight_decay=CFG['weight_decay'])
    warmup_sch  = optim.lr_scheduler.LambdaLR(
        optimizer, lr_lambda=make_warmup_lambda(CFG['warmup_epochs']))
    plateau_sch = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=CFG['lr_factor'],
        patience=CFG['lr_patience'], min_lr=CFG['min_lr'], verbose=False)
    scaler      = torch.amp.GradScaler('cuda', enabled=use_amp)

    # ── Training state ───────────────────────────────────────────────
    best_val      = float('inf')
    no_improve    = 0
    nan_total     = 0
    history       = dict(train_main=[], train_aux=[], val_main=[], val_aux=[], lr=[])

    print(f"\n[Train]  {CFG['max_epochs']} ep | "
          f"eff-batch {CFG['micro_batch']*CFG['accum_steps']:,} | "
          f"lr={CFG['lr']:.0e} | warmup={CFG['warmup_epochs']} | clip={CFG['grad_clip']}\n")
    t0 = time.time()

    for epoch in range(1, CFG['max_epochs'] + 1):

        # ── TRAIN ────────────────────────────────────────────────────
        model.train()
        main_sum = aux_sum = nan_steps = 0

        pbar = tqdm(enumerate(train_loader), total=len(train_loader),
                    desc=f"Ep {epoch:03d}/{CFG['max_epochs']}", leave=False)
        optimizer.zero_grad()

        for step, (xb, tb, ab) in pbar:
            xb = xb.to(device, non_blocking=True)
            tb = tb.to(device, non_blocking=True)
            ab = ab.to(device, non_blocking=True)

            with torch.amp.autocast('cuda', enabled=use_amp):
                preds              = model(xb)
                loss, l_main, l_aux = criterion(preds, tb, ab)

            # NaN FIX 2: catch NaN loss before backward
            if is_bad(loss):
                nan_steps += 1
                optimizer.zero_grad()
                scaler.update()
                continue

            scaler.scale(loss / CFG['accum_steps']).backward()
            main_sum += l_main.item()
            aux_sum  += l_aux.item()

            is_last  = (step + 1) == len(train_loader)
            is_accum = (step + 1) % CFG['accum_steps'] == 0

            if is_accum or is_last:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip'])

                # NaN FIX 1: detect AMP skip via scale drop
                scale_before = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()

                if scaler.get_scale() < scale_before:
                    # AMP skipped the step — weights are safe but grads are bad
                    nan_steps += 1
                    optimizer.zero_grad()
                    load_best(model, CFG['checkpoint_path'], device)  # NaN FIX 5
                else:
                    optimizer.zero_grad()

            if step % 100 == 0:
                pbar.set_postfix(Main=f'{l_main.item():.4f}',
                                 Aux=f'{l_aux.item():.4f}',
                                 NaN=nan_steps)

        if nan_steps:
            nan_total += nan_steps
            print(f"  ⚠  {nan_steps} NaN steps skipped  (cumulative: {nan_total})")

        valid = max(len(train_loader) - nan_steps, 1)
        avg_tr_main = main_sum / valid
        avg_tr_aux  = aux_sum  / valid

        # ── VALIDATE ─────────────────────────────────────────────────
        model.eval()
        vm_sum = va_sum = bad_batches = 0

        with torch.no_grad():
            for xv, tv, av in val_loader:
                xv = xv.to(device, non_blocking=True)
                tv = tv.to(device, non_blocking=True)
                av = av.to(device, non_blocking=True)
                with torch.amp.autocast('cuda', enabled=use_amp):
                    vp, (_, v_main, v_aux) = model(xv), criterion(model(xv), tv, av)
                if is_bad(v_main):
                    bad_batches += 1
                    continue
                vm_sum += v_main.item()
                va_sum += v_aux.item()

        # NaN FIX 5: full val NaN → roll back, skip epoch
        if bad_batches == len(val_loader):
            print(f"  ✗  Entire val NaN — rolling back to best checkpoint")
            load_best(model, CFG['checkpoint_path'], device)
            continue

        valid_val    = max(len(val_loader) - bad_batches, 1)
        avg_val_main = vm_sum / valid_val
        avg_val_aux  = va_sum / valid_val
        current_lr   = optimizer.param_groups[0]['lr']

        # ── SCHEDULERS ───────────────────────────────────────────────
        if epoch <= CFG['warmup_epochs']:
            warmup_sch.step()
        else:
            plateau_sch.step(avg_val_main)

        # ── HISTORY & LOG ────────────────────────────────────────────
        history['train_main'].append(avg_tr_main)
        history['train_aux' ].append(avg_tr_aux)
        history['val_main'  ].append(avg_val_main)
        history['val_aux'   ].append(avg_val_aux)
        history['lr'        ].append(current_lr)

        print(f"Ep {epoch:03d} | "
              f"Tr {avg_tr_main:.5f}/{avg_tr_aux:.5f} | "
              f"Val {avg_val_main:.5f}/{avg_val_aux:.5f} | "
              f"LR {current_lr:.2e} | "
              f"{(time.time()-t0)/60:.1f} min")

        # ── CHECKPOINT + EARLY STOPPING ──────────────────────────────
        if avg_val_main < best_val:
            best_val   = avg_val_main
            no_improve = 0
            torch.save(model.state_dict(), CFG['checkpoint_path'])
            print(f"  ✦ New best {best_val:.5f} → saved")
        else:
            no_improve += 1
            print(f"  · {no_improve}/{CFG['patience']}")

        if no_improve >= CFG['patience']:
            print(f"\n[Stop]  Early stop at ep {epoch}  best={best_val:.5f}")
            break

    # ── SUMMARY ──────────────────────────────────────────────────────
    save_plot(history, CFG['history_path'])
    print(f"\n{'='*65}")
    print(f"  ✅  DONE  |  Best val main: {best_val:.6f}  |  "
          f"NaN recoveries: {nan_total}  |  "
          f"Time: {(time.time()-t0)/60:.1f} min")
    print(f"{'='*65}\n")

    # ── VERIFY FROM DISK — never from in-memory model (NaN FIX 5) ────
    model_best = TitanSciML(CFG['n_global'], CFG['n_spatial'], CFG['width']).to(device)
    model_best.load_state_dict(
        torch.load(CFG['checkpoint_path'], map_location=device, weights_only=True))
    model_best.eval()
    xv, tv, av = next(iter(val_loader))
    with torch.no_grad():
        out = model_best(xv.to(device))
    assert not torch.isnan(out).any(), "CHECKPOINT HAS NaN — training failed"
    print(f"[Verify]  shape={out.shape}  range=[{out.min():.3f}, {out.max():.3f}]")
    print("[Verify]  ✓  Clean checkpoint ready for inference.")


if __name__ == "__main__":
    main()


  TITAN V2  ·  JET IMPINGEMENT SciML  (NaN-hardened)
[Device]  cuda  |  AMP: True
[Data]  16,600,000 samples  |  in=12  tgt=5
[Data]  Train: 14,940,000  Val: 1,660,000
[Model]  7,348,743 params  (7.35 M)

[Train]  200 ep | eff-batch 8,192 | lr=2e-04 | warmup=5 | clip=0.5

